In [1]:
import cv2
import mediapipe as mp
import uuid
import time
import os

In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

CONFIGURATION BLOCK


In [16]:
CAMERA_INDEX = 1
START_DELAY = 2
SESSION_IMAGES = 200 #PER ONE SESSION
CLASS_IMAGES = 5000 #PER ONE CLASS
CAPTURE_INTERVAL = 0.1
DELAY = 5

SAVE_PATH = "../datasets/raw/"

In [17]:
CLASSES = ["Dragon", "Tiger", "Dog", "Rat", "Ram", "Horse", "Monkey", "Boar", "Bird", "Serpent", "Ox", "Hare"]

print(CLASSES[1])

Tiger


In [19]:
i = 0 # Change this index to select different class from CLASSES list
SAVE_DIR = os.path.join(SAVE_PATH, CLASSES[i])

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR, exist_ok=True)

if len(os.listdir(SAVE_DIR)) >= CLASS_IMAGES:
    print(f"Class {CLASSES[i]} already has {CLASS_IMAGES} images. Skipping...\n")
    print("Change the class name or increase the CLASS_IMAGES limit if you want to collect more data.")

else:

    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)  # Replace with your camera device ID)
    
    total_count = len(os.listdir(SAVE_DIR))
    saved_count = 0 
    start_time = time.time()
    last_capture_time = 0

    with mp_hands.Hands(
        min_detection_confidence=0.3, min_tracking_confidence=0.3
    ) as hands:
        
        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            elapsed = time.time() - start_time
            remaining = max(0, int(START_DELAY - elapsed))

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frame = cv2.flip(frame, 1)

            frame.flags.writeable = False

            results = hands.process(frame)

            frame.flags.writeable = True

            # if results.multi_hand_landmarks:
            #     for  hand_landmarks in results.multi_hand_landmarks:
            #         mp_drawing.draw_landmarks(frame, hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS)
            
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

            if elapsed < START_DELAY:
                cv2.putText(
                    frame,
                    f"Starting in {remaining} seconds...",
                    (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    2,
                )

            else:

                current_time = time.time()

                if current_time - last_capture_time >= CAPTURE_INTERVAL:

                    filename = os.path.join(
                        SAVE_DIR,
                        f"{total_count}_{uuid.uuid4()}.jpg"
                    )

                    cv2.imwrite(filename, frame)

                    saved_count += 1
                    total_count += 1
                    last_capture_time = current_time

                cv2.putText(
                    frame,
                    f"Captured: {saved_count}/{SESSION_IMAGES}",
                    (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Class: {CLASSES[i]}",
                    (20, 100),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 0, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Total Images in Class: {len(os.listdir(SAVE_DIR))}/{CLASS_IMAGES}",
                    (20, 150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 0),
                    2
                )

            cv2.imshow("Hand Tracking", frame)

            if saved_count >= SESSION_IMAGES:
                print(f"Saved {SESSION_IMAGES} images.")
                break

            # Manual exit
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()